# Logistic Regression (the simple model)

**Question:** How does the simple model work, how well does it do, and what does it say about each genre?

The details and dates are in this role's GitHub issue. Keep your work inside this folder.

In [1]:
import sys; sys.path.insert(0, "../..")
import pandas as pd
from common import load, logistic_regression, FEATURES, GENRES

d = load()                      # d.X_train, d.y_train, d.X_test, d.y_test, d.cv (shared folds)
model = logistic_regression()   # the frozen version everyone compares
preds = pd.read_csv("../../predictions/logistic_regression.csv")   # its saved test-set predictions
print(len(d.X_train), "train songs,", len(d.X_test), "test songs")

36037 train songs, 9009 test songs


## 1. The data at a glance

45,046 unique songs, 14 Spotify audio features, 5 parent genres. The split is frozen in `clean_songs.csv`, and the 5 training folds are grouped by artist, so the model is always validated on artists it hasn't seen.

In [3]:
print("train songs:", len(d.X_train), "  test songs:", len(d.X_test), "  total:", len(d.songs))
print("\n genre counts (train):")
print(d.y_train.value_counts())
print("\n genre counts (test):")
print(d.y_test.value_counts())
print("\n fold sizes (train):")
print(d.songs[d.songs["split"] == "train"].groupby("fold").size())
print("\n missing feature values:", int(d.songs[FEATURES].isna().sum().sum()))

train songs: 36037   test songs: 9009   total: 45046

 genre counts (train):
broad_genre
electronic    10920
rock           9529
pop            8165
latin          5898
hiphop/rnb     1525
Name: count, dtype: int64

 genre counts (test):
broad_genre
electronic    2730
rock          2383
pop           2041
latin         1474
hiphop/rnb     381
Name: count, dtype: int64

 fold sizes (train):
fold
0    7208
1    7208
2    7207
3    7207
4    7207
dtype: int64

 missing feature values: 0


## 2. The dumb baseline

Before trusting any model, the bar it has to beat: always guess the most common genre. In train that's electronic, about 30% of songs — so a model that can't beat ~30% on test has learned nothing.

In [5]:
majority = d.y_train.value_counts().idxmax()
baseline_acc = (d.y_test == majority).mean()
print("majority class in train:", majority, f"({(d.y_train == majority).mean():.1%} of train)")
print(f"baseline test accuracy (always guess {majority}): {baseline_acc:.4f}")

majority class in train: electronic (30.3% of train)
baseline test accuracy (always guess electronic): 0.3030


## 3. How the frozen model does on the test songs

Test results come from the saved predictions (`predictions/logistic_regression.csv`) — the frozen model's output, per the issue. Accuracy 57% vs the 30% baseline: the features clearly carry genre signal. But the per-genre view is the real story: electronic, rock and latin are all above 55% recall, pop is middling, and hip-hop/R&B is barely detected at all (recall 10%).

In [7]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

y_true, y_pred = preds["broad_genre"], preds["predicted"]
print(f"test accuracy: {accuracy_score(y_true, y_pred):.4f}   macro-F1: {f1_score(y_true, y_pred, average='macro'):.4f}")
print()
print(classification_report(y_true, y_pred, digits=3))
print("confusion matrix (rows = actual, cols = predicted)")
print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=GENRES), index=GENRES, columns=GENRES))

test accuracy: 0.5695   macro-F1: 0.4886

              precision    recall  f1-score   support

  electronic      0.664     0.683     0.674      2730
  hiphop/rnb      0.404     0.100     0.160       381
       latin      0.569     0.552     0.560      1474
         pop      0.432     0.459     0.445      2041
        rock      0.589     0.621     0.604      2383

    accuracy                          0.570      9009
   macro avg      0.532     0.483     0.489      9009
weighted avg      0.565     0.570     0.563      9009

confusion matrix (rows = actual, cols = predicted)
            electronic  hiphop/rnb  latin  pop  rock
electronic        1865          18     76  339   432
hiphop/rnb          89          38     71  147    36
latin              109          13    813  432   107
pop                355          10    283  936   457
rock               390          15    187  312  1479


## 4. Sanity check: does refitting the frozen model reproduce the saved file?

Yes — exactly. Fitting `logistic_regression()` on the train split and predicting test gives the identical predictions (100% agreement), so the saved file really is the frozen model's output and the test numbers above are trustworthy.

In [9]:
model.fit(d.X_train, d.y_train)
my_pred = model.predict(d.X_test)
print(f"my refit test accuracy: {accuracy_score(d.y_test, my_pred):.4f}")
print(f"agreement with saved predictions: {(my_pred == y_pred.values).mean():.4f}")
print(f"matches saved file exactly: {(my_pred == y_pred.values).all()}")

my refit test accuracy: 0.5695
agreement with saved predictions: 1.0000
matches saved file exactly: True


## 5. Does regularization strength (C) matter?

Tuned on the training folds only, never on test. Short answer: no. CV accuracy moves from 56.5% (C=0.01) to 56.6% (C=1.0) and stays there — a 0.15-point range, far inside the ~1.4-point fold-to-fold noise. The frozen default C=1.0 is already at the peak, so there's nothing to gain by tuning.

In [11]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

for C in [0.01, 0.1, 1.0, 10.0, 100.0]:
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=C))
    s = cross_val_score(m, d.X_train, d.y_train, cv=d.cv, scoring="accuracy", n_jobs=-1)
    print(f"  C={C:<6} CV accuracy {s.mean():.4f} +/- {s.std():.4f}   folds {[round(float(x), 4) for x in s]}")
print()
print("best C on CV: 1.0 (the frozen default)")

  C=0.01   CV accuracy 0.5650 +/- 0.0134   folds [0.5459, 0.5813, 0.5572, 0.5613, 0.5792]
  C=0.1    CV accuracy 0.5663 +/- 0.0139   folds [0.5463, 0.583, 0.5577, 0.5635, 0.5808]
  C=1.0    CV accuracy 0.5665 +/- 0.0139   folds [0.5466, 0.5828, 0.5578, 0.5638, 0.5814]
  C=10.0   CV accuracy 0.5664 +/- 0.0139   folds [0.5465, 0.5827, 0.5578, 0.5639, 0.5814]
  C=100.0  CV accuracy 0.5664 +/- 0.0139   folds [0.5465, 0.5827, 0.5578, 0.5639, 0.5814]

best C on CV: 1.0 (the frozen default)


## 6. Reading the model: which features push a song toward each genre

This is the simple model's superpower — it's readable. Coefficients are on standardized features, so magnitudes are comparable. A positive coefficient means higher values of that feature push the song toward that genre (holding the others fixed); negative means the opposite.

- **Electronic:** instrumental (+0.90) and danceable (+0.56), low valence (−0.59) and non-acoustic (−0.47). Instrumental dance music that doesn't sound "happy".
- **Hip-hop/R&B:** loud (+0.57), danceable (+0.42), speechy (+0.21), explicit (+0.21) — but low energy (−0.78). Words over beats, not intensity.
- **Latin:** acoustic (+0.77), high valence (+0.57), live-sounding (+0.43). The brightest, most acoustic genre.
- **Pop:** nothing pushes hard — the biggest coefficient is mode at +0.13. Pop has no signature, which is exactly why it's the genre everything gets confused with.
- **Rock:** high energy (+0.58) and *not* danceable — danceability −1.03 is the single strongest coefficient in the whole model.

In [13]:
import numpy as np
lr = model.named_steps["logisticregression"]
for i, g in enumerate(GENRES):
    order = np.argsort(lr.coef_[i])
    top_pos = [(FEATURES[j], round(float(lr.coef_[i][j]), 2)) for j in order[::-1][:5]]
    top_neg = [(FEATURES[j], round(float(lr.coef_[i][j]), 2)) for j in order[:5]]
    print(f"{g}:")
    print("  pushes TOWARD:", ", ".join(f"{f} ({v:+.2f})" for f, v in top_pos))
    print("  pushes AWAY:  ", ", ".join(f"{f} ({v:+.2f})" for f, v in top_neg))

electronic:
  pushes TOWARD: instrumentalness (+0.90), danceability (+0.56), energy (+0.28), tempo (+0.13), time_signature (+0.05)
  pushes AWAY:   valence (-0.59), acousticness (-0.47), loudness (-0.16), liveness (-0.13), explicit (-0.07)
hiphop/rnb:
  pushes TOWARD: loudness (+0.57), danceability (+0.42), speechiness (+0.21), explicit (+0.21), duration_ms (+0.11)
  pushes AWAY:   energy (-0.78), instrumentalness (-0.45), mode (-0.13), time_signature (-0.05), liveness (-0.01)
latin:
  pushes TOWARD: acousticness (+0.77), valence (+0.57), liveness (+0.43), duration_ms (+0.13), loudness (+0.12)
  pushes AWAY:   instrumentalness (-0.32), explicit (-0.14), speechiness (-0.09), mode (-0.01), tempo (+0.01)
pop:
  pushes TOWARD: mode (+0.13), speechiness (+0.06), acousticness (+0.05), danceability (+0.03), tempo (+0.02)
  pushes AWAY:   instrumentalness (-0.32), explicit (-0.23), liveness (-0.19), energy (-0.11), loudness (-0.05)
rock:
  pushes TOWARD: energy (+0.58), explicit (+0.24), instr

## 7. Takeaways

- **It works, clearly:** 57% test accuracy vs a 30% always-guess-electronic baseline (+27 points). Macro-F1 0.49.
- **The headline weakness is hip-hop/R&B:** recall 10% — only 38 of 381 test songs caught, most called pop (147). It's 4.2% of train and has no clean feature signature, so the model almost never predicts it (94 predicted vs 381 actual).
- **Pop is the mushy middle:** no feature pushes strongly toward pop (max coefficient +0.13), so borderline songs land there — and pop bleeds into rock (457) and latin (283).
- **Electronic ↔ rock is the other big mix-up** (432 + 390 songs swapped).
- **Tuning buys nothing:** C from 0.01 to 100 moves CV accuracy by 0.15 points, inside fold noise. The frozen C=1.0 is already best.
- **Why logistic regression is "simple":** one weight per feature per genre, all readable — the whole model fits in the coefficient table above, and it runs in under a second.